# NB_Bounds: dose, detection limit and heat-transfer screens
Illustrative scenario calculations, not predictions. The share of particles that strike a tube, the exposure time and the radiative retention are **assumed** and labeled as such. Equal integrated dose is not equal damage at a different impact speed. Run all cells.

In [ ]:
import math
# Inputs (source in the comment; 'assumed' = our scenario)
R,P,M = 8.314462,101325.0,0.028013                  # N2 ideal gas
Tgas  = 1561.15                                     # K, flue gas near the first rows (Janicka 2023, Table 4, 1288 C), N2 as the carrier gas
solids= (0.31,0.45)                                 # wt fraction solids, Janicka 2023, Table 3
U     = (5.8,10.8)                                  # m/s gas velocity, Janicka 2023, Table 3
eta_imp=(0.1,1.0); eta_c=0.3                        # share of particles striking the tube: assumed bounds and central case
hours = 8000; yr = hours*3600.0                     # assumed full utility year
rho_steel = 7840.0                                  # kg/m3
rho_g = P*M/(R*Tgas)
c_lo,c_hi = rho_g*solids[0]/(1-solids[0]), rho_g*solids[1]/(1-solids[1])   # particle mass concentration kg/m3
print(f'gas density {rho_g:.3f} kg/m3; particle concentration {c_lo:.3f} to {c_hi:.3f} kg/m3')
c_mid,U_mid = 0.5*(c_lo+c_hi), 8.3
slope = lambda eta_,c,u: eta_*c*u*yr/rho_steel*1000      # mm/yr of wall loss per unit specific erosion rate E (g/g)
Ecrit = lambda s: 0.25/s                                  # E at which wall loss reaches 0.25 mm/yr
print('E for 0.25 mm/yr, central: %.2e; bounds %.1e to %.1e'%(Ecrit(slope(eta_c,c_mid,U_mid)),Ecrit(slope(1.0,c_hi,U[1])),Ecrit(slope(0.1,c_lo,U[0]))))
# laboratory air-jet flux (Akbarzadeh 2012): 2.0 g/min through a 1.5 mm nozzle
flux_lab=(2.0e-3/60)/(math.pi*(0.75e-3)**2)
dose_1um = lambda E: 1e-6*rho_steel/E                     # kg/m2 of particles to remove 1 micrometer
for E in (2e-7,3.5e-8,1e-9):
    d=dose_1um(E); print(f'E={E:.1e}: dose for 1 um = {d:.2e} kg/m2 = {d/flux_lab/3600:.2f} h of jet time')
flux_b = eta_c*c_mid*0.5*sum(U)
print(f'assumed-scenario dose per boiler year {flux_b*yr:.2e} kg/m2 = {flux_b*yr/flux_lab/3600:.0f} h of jet time (equal dose is not equal damage)')

In [ ]:
# Heat-transfer screens for a thin bonded layer on a tube (downstream superheater coefficients, Janicka 2023, Table 5; NOT first-row values)
hc,hr = 46.0,242.0
def conduction_loss(t,k,Rc=0): b=(hc+hr)*(t/k+Rc); return 100*b/(1+b)
Q = lambda s,t=2e-4,k=10: ((1/(hc+hr))/((1/(hc+s*hr))+t/k)-1)*100   # s = assumed fraction of the radiative coefficient that survives
print('conduction only, 0.2 mm: k=10 -> %.3f %%, k=9 -> %.3f %%'%(conduction_loss(2e-4,10),conduction_loss(2e-4,9)))
print('lowest k that keeps conduction loss under 5 %%: %.3f W/m/K'%((hc+hr)*2e-4*0.95/0.05))
for s in (0.7,0.9,1.0,1.1): print(f's={s}: heat duty change {Q(s):+.2f} %  (s is an assumed retention, not a measured emissivity)')
assert abs(conduction_loss(2e-4,9)-0.636)<0.003 and abs(Q(0.9)+8.884)<0.01